In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_descriptives_gsheet.py
==========================
Colab runner: FULL DESCRIPTIVE ANALYSIS of every graph in your Google Sheet.

Works on your merged sheet layout: ONE ROW PER GRAPH with columns
link | graph | type. For every row it parses the graph and computes the
complete structural profile:
sizes, degrees, cycles (and whether a recipe cycle is a LEGAL Repeat loop-back),
longest path, linearity, evidence coverage, certainty mix, node/primitive
composition, outcome presence, LA-CFG checks, and the hard-fail gate flags.

Results are written INTO THE SAME ROWS, in the columns immediately after your
data (headers added on row 1), in a single batched update. Rows whose graph
cell holds an extractor error like {"error": "..."} get status extractor_error
and blank metrics. Nothing in columns A-C is ever touched.

Recipe schema note: node types are read from the `primitive` key with `type`
as fallback, matching what the extractor actually produces.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("gspread", "gspread"), ("json_repair", "json-repair")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _pip([_pkg])

import json, re
from collections import Counter, defaultdict
import gspread, json_repair

# =====================================================================
# 1. CONFIG
# =====================================================================
SHEET_URL = "https://docs.google.com/spreadsheets/d/177xqn4i1s6dYVYueV0P0vc6_NIbn_-mRLLqvHIGVsj4/edit?usp=sharing"

# The merged tab: one row per graph, columns found BY HEADER.
TAB_NAME    = None  # set to your tab's exact name (or None = first tab)
LINK_HEADER  = "link"
GRAPH_HEADER = "graph"
TYPE_HEADER  = "type"         # values like "trajectory" / "recipe" (case-insensitive)

# =====================================================================
# 2. OPEN SHEET
# =====================================================================
def open_spreadsheet():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    u = SHEET_URL.strip()
    if "PASTE_YOUR" in u:
        raise SystemExit("Set SHEET_URL to your Google Sheet's URL.")
    if "drive.google.com/file" in u:
        raise SystemExit("That is a Drive FILE link. Use the docs.google.com/spreadsheets/d/... URL.")
    sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    print(f"[sheet] opened '{sh.title}'")
    return sh

# =====================================================================
# 3. GRAPH PARSING + METRICS
# =====================================================================
def parse_graph(cell):
    s = re.sub(r"^```(?:json)?|```$", "", str(cell or "").strip(), flags=re.M).strip()
    if not s.startswith("{"):
        return None
    try:
        g = json.loads(s)
    except Exception:
        try:
            g = json_repair.loads(s)
        except Exception:
            return None
    return g if isinstance(g, dict) else None

def node_type(n):
    return str(n.get("primitive") or n.get("type") or "?")

def edge_ends(e):
    return e.get("source", e.get("from")), e.get("target", e.get("to"))

FP_PAST = re.compile(r"\b(i|we)\s+(was|were|did|applied|emailed|built|moved|quit|started|"
                     r"finished|joined|got|went|paid|lost|wrote|submitted|received|passed)\b", re.I)
LEGAL_PRIMS = {"Op", "Branch", "AND", "OR", "NOT", "Repeat"}

def find_back_edges(nodes, edges):
    """DFS: returns list of back edges (u,v) that close cycles."""
    ids = {n.get("id") for n in nodes}
    adj = defaultdict(list)
    for e in edges:
        s, t = edge_ends(e)
        if s in ids and t in ids:
            adj[s].append(t)
    color, back = {i: 0 for i in ids}, []
    def dfs(u):
        color[u] = 1
        for v in adj[u]:
            if color.get(v) == 1:
                back.append((u, v))
            elif color.get(v) == 0:
                dfs(v)
        color[u] = 2
    for i in list(ids):
        if color[i] == 0:
            dfs(i)
    return back

def longest_path(nodes, edges, exclude=frozenset()):
    """Longest path in edges (back-edges excluded) via topological DP."""
    ids = {n.get("id") for n in nodes}
    adj, indeg = defaultdict(list), {i: 0 for i in ids}
    for e in edges:
        s, t = edge_ends(e)
        if s in ids and t in ids and (s, t) not in exclude:
            adj[s].append(t); indeg[t] += 1
    from collections import deque
    qq = deque([i for i in ids if indeg[i] == 0])
    dist = {i: 0 for i in ids}
    seen = 0
    while qq:
        u = qq.popleft(); seen += 1
        for v in adj[u]:
            dist[v] = max(dist[v], dist[u] + 1)
            indeg[v] -= 1
            if indeg[v] == 0:
                qq.append(v)
    if seen < len(ids):        # residual cycle even after excluding back edges
        return None
    return max(dist.values()) if dist else 0

def describe(g, gtype):
    """Full descriptive profile of one graph. gtype in {'trajectory','recipe'}."""
    nodes, edges = g.get("nodes", []) or [], g.get("edges", []) or []
    n, e = len(nodes), len(edges)
    ids = [x.get("id") for x in nodes]
    idset = set(ids)
    m = {"n_nodes": n, "n_edges": e}
    m["duplicate_node_ids"] = len(ids) - len(idset)
    m["density"] = round(e / (n * (n - 1)), 4) if n > 1 else 0

    outdeg, indeg = Counter(), Counter()
    dangling = 0
    for ed in edges:
        s, t = edge_ends(ed)
        if s in idset and t in idset:
            outdeg[s] += 1; indeg[t] += 1
        else:
            dangling += 1
    m["dangling_edges"] = dangling
    m["max_out_degree"] = max(outdeg.values()) if outdeg else 0
    m["max_in_degree"] = max(indeg.values()) if indeg else 0
    m["isolated_nodes"] = sum(1 for i in idset if outdeg[i] == 0 and indeg[i] == 0)

    back = find_back_edges(nodes, edges)
    m["has_cycle"] = bool(back)
    m["back_edges"] = len(back)

    # legal Repeat loop-back: the CYCLE (not the arbitrary DFS back edge) is legal
    # when removing every edge that leaves a Repeat-with-stop_condition node
    # makes the graph acyclic.
    node_by = {x.get("id"): x for x in nodes}
    repeat_out = frozenset(
        (edge_ends(ed)) for ed in edges
        if node_type(node_by.get(edge_ends(ed)[0], {})) == "Repeat"
        and str(node_by.get(edge_ends(ed)[0], {}).get("stop_condition", "")).strip())
    residual = find_back_edges(nodes, [ed for ed in edges if edge_ends(ed) not in repeat_out])
    legal = bool(back) and gtype == "recipe" and not residual
    m["legal_repeat_cycle"] = legal

    lp = longest_path(nodes, edges, exclude=(repeat_out | frozenset(residual)) if gtype == "recipe"
                      else frozenset(back))
    m["longest_path_edges"] = lp if lp is not None else ""
    m["linearity"] = round(lp / e, 3) if (lp not in (None, "") and e) else ""

    ev = [str(x.get("evidence", "")).strip() for x in nodes]
    with_ev = [q for q in ev if len(q) >= 10]
    m["evidence_coverage"] = round(len(with_ev) / n, 3) if n else 0
    m["avg_evidence_chars"] = round(sum(len(q) for q in with_ev) / len(with_ev), 1) if with_ev else 0

    cert = Counter(str(x.get("certainty", "")).strip().lower() for x in nodes)
    m["certainty_stated"] = cert.get("stated", 0)
    m["certainty_implied"] = cert.get("implied", 0)
    m["certainty_other"] = n - m["certainty_stated"] - m["certainty_implied"]
    m["unknowns_count"] = len(g.get("unknowns", []) or [])
    m["goal_present"] = bool(str(g.get("goal", "")).strip())

    tc = Counter(node_type(x) for x in nodes)
    m["type_counts_json"] = json.dumps(dict(tc), ensure_ascii=False)

    gates = []
    if gtype == "trajectory":
        m["start_state_count"] = tc.get("start_state", 0)
        m["outcome_count"] = tc.get("outcome", 0)
        rels = Counter(str(ed.get("relation", "")) for ed in edges)
        m["retry_edges"] = rels.get("retry_of", 0)
        m["logic_edge_leak"] = rels.get("enabled", 0) + rels.get("alternative", 0)
        if m["has_cycle"]: gates.append("cycle")
        if m["start_state_count"] != 1: gates.append("start_state_count!=1")
        if m["outcome_count"] < 1: gates.append("missing_outcome")
        if m["logic_edge_leak"]: gates.append("logic_edge_leak")
        # recipe-only columns stay blank
        m.update(ungated_edges="", repeats_total="", repeats_missing_stop="",
                 branches_total="", branch_issues="", illegal_primitives="",
                 personal_log_leak_nodes="")
    else:
        m.update(start_state_count="", outcome_count="", retry_edges="", logic_edge_leak="")
        m["ungated_edges"] = sum(1 for ed in edges if not str(ed.get("condition", "")).strip())
        reps = [i for i, x in node_by.items() if node_type(x) == "Repeat"]
        m["repeats_total"] = len(reps)
        m["repeats_missing_stop"] = sum(
            1 for i in reps if not str(node_by[i].get("stop_condition", "")).strip())
        brs = [i for i, x in node_by.items() if node_type(x) == "Branch"]
        m["branches_total"] = len(brs)
        issues = []
        for i in brs:
            outs = [ed for ed in edges if edge_ends(ed)[0] == i]
            conds = [str(ed.get("condition", "")).strip().lower() for ed in outs]
            if len(outs) < 2: issues.append(f"{i}:<2edges")
            if not any(c in ("else", "otherwise", "fallback", "default") for c in conds):
                issues.append(f"{i}:no_fallback")
        m["branch_issues"] = ";".join(issues)
        illegal = sorted({node_type(x) for x in nodes} - LEGAL_PRIMS - {"?"})
        m["illegal_primitives"] = ";".join(illegal)
        m["personal_log_leak_nodes"] = sum(
            1 for x in nodes if FP_PAST.search(f"{x.get('label','')} {x.get('description','')}"))
        if m["has_cycle"] and not legal: gates.append("cycle_not_repeat_gated")
        if illegal: gates.append("illegal_primitives")
        if m["repeats_missing_stop"]: gates.append("repeat_missing_stop")
        if issues: gates.append("branch_missing_fallback_or_arity")
        if m["ungated_edges"]: gates.append("ungated_edges")
        if m["personal_log_leak_nodes"] > 2: gates.append("personal_log_leakage")
    if m["evidence_coverage"] < 0.8 and n: gates.append("evidence_coverage<0.8")
    if m["duplicate_node_ids"]: gates.append("duplicate_node_ids")
    if m["dangling_edges"]: gates.append("dangling_edges")
    if n == 0: gates.append("empty_graph")
    m["gates"] = ";".join(gates)
    return m

# =====================================================================
# 4. COLUMN ORDER FOR THE OUTPUT TAB
# =====================================================================
COLUMNS = ["status",
    "n_nodes", "n_edges", "density", "max_out_degree", "max_in_degree",
    "isolated_nodes", "dangling_edges", "duplicate_node_ids",
    "has_cycle", "back_edges", "legal_repeat_cycle",
    "longest_path_edges", "linearity",
    "evidence_coverage", "avg_evidence_chars",
    "certainty_stated", "certainty_implied", "certainty_other",
    "unknowns_count", "goal_present",
    "start_state_count", "outcome_count", "retry_edges", "logic_edge_leak",
    "ungated_edges", "repeats_total", "repeats_missing_stop",
    "branches_total", "branch_issues", "illegal_primitives", "personal_log_leak_nodes",
    "type_counts_json", "gates"]

# =====================================================================
# 5. MAIN
# =====================================================================
def main():
    sh = open_spreadsheet()
    ws = sh.worksheet(TAB_NAME) if TAB_NAME else sh.get_worksheet(0)
    rows = ws.get_all_values()
    if not rows:
        raise SystemExit("Tab is empty.")
    header = [h.strip().lower() for h in rows[0]]
    def col(name):
        return header.index(name) if name in header else None
    li, gi, ty = col(LINK_HEADER), col(GRAPH_HEADER), col(TYPE_HEADER)
    if gi is None or ty is None:
        raise SystemExit(f"Could not find '{GRAPH_HEADER}' / '{TYPE_HEADER}' headers in row 1: {rows[0]}")
    li = 0 if li is None else li
    start_col = len(rows[0])          # first empty column after her data
    print(f"[cols] link @ {li+1} | graph @ {gi+1} | type @ {ty+1} | "
          f"descriptives will start at column {start_col+1}")

    out_matrix = [COLUMNS]            # header row for the new columns
    results = []
    for r in rows[1:]:
        cellv = r[gi] if gi < len(r) else ""
        gtype = (r[ty] if ty < len(r) else "").strip().lower()
        gtype = "trajectory" if gtype.startswith("traj") else ("recipe" if gtype.startswith("rec") else gtype)
        row_out = {c: "" for c in COLUMNS}
        if not str(cellv).strip():
            row_out["status"] = "empty_cell"
        else:
            g = parse_graph(cellv)
            if g is None:
                row_out["status"] = "parse_failed"
            elif "error" in g and not g.get("nodes"):
                row_out["status"] = f"extractor_error: {str(g.get('error'))[:80]}"
            elif not g.get("nodes"):
                row_out["status"] = "empty_graph"
            elif gtype not in ("trajectory", "recipe"):
                row_out["status"] = f"unknown_type: {gtype[:30]}"
            else:
                row_out["status"] = "ok"
                row_out.update(describe(g, gtype))
        results.append({"type": gtype, **row_out})
        out_matrix.append([("" if row_out.get(c) is None else
                            ("TRUE" if row_out.get(c) is True else
                             "FALSE" if row_out.get(c) is False else str(row_out.get(c, ""))))
                           for c in COLUMNS])

    # single batched write into the columns right after her data
    def a1(idx0):
        sname, nn = "", idx0 + 1
        while nn:
            nn, rr = divmod(nn - 1, 26)
            sname = chr(65 + rr) + sname
        return sname
    rng = f"{a1(start_col)}1"
    ws.update(rng, out_matrix, value_input_option="RAW")
    ok = sum(1 for r in results if r["status"] == "ok")
    print(f"[done] wrote {len(COLUMNS)} descriptive columns for {len(results)} rows "
          f"({ok} ok) starting at column {a1(start_col)}")

    from statistics import median
    for gtype in ("trajectory", "recipe"):
        sub = [r for r in results if r["type"] == gtype and r["status"] == "ok"]
        if not sub: continue
        med = lambda k: median([float(r[k]) for r in sub if str(r.get(k, "")) not in ("", "None")])
        cyc = sum(1 for r in sub if r.get("has_cycle") is True)
        legal = sum(1 for r in sub if r.get("legal_repeat_cycle") is True)
        lin = [float(r["linearity"]) for r in sub if str(r.get("linearity","")) != ""]
        lin_med = f"{median(lin):.2f}" if lin else "-"
        print(f"[{gtype}] n={len(sub)} | median nodes {med('n_nodes'):.0f} | "
              f"median linearity {lin_med} | cycles {cyc}"
              + (f" (legal Repeat: {legal})" if gtype == "recipe" else ""))

if __name__ == "__main__":
    main()

[sheet] opened 'merged_data'
[cols] link @ 1 | graph @ 2 | type @ 3 | descriptives will start at column 4


/tmp/ipykernel_2025/1256803475.py:319: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, out_matrix, value_input_option="RAW")


[done] wrote 34 descriptive columns for 240 rows (235 ok) starting at column D
[trajectory] n=116 | median nodes 24 | median linearity 0.94 | cycles 9
[recipe] n=119 | median nodes 30 | median linearity 0.45 | cycles 24 (legal Repeat: 11)


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_evaluation_merged_gsheet.py
===============================
Colab evaluation runner for the MERGED sheet layout (one row per graph:
link | graph | type), judge = DeepSeek-V4-Pro, prompt = prompt_judge_deepseek_v2.txt.

How it works:
  - Groups rows by link. A link with both a trajectory row and a recipe row is
    class "mixed" (coherence gets scored); one row only -> that class.
  - ONE judge call per link covers both its graphs, then the results are written
    back INTO EACH ROW, in the columns right after your existing data (headers
    added on row 1). Columns A-C (and any descriptive columns) are never touched.
  - Every rubric label is stored per row: the six criterion scores of that row's
    rubric, notes+spans JSON, composite (TQS or RQS), gates, over-reach, verdicts,
    link-level coherence, and the full judge JSON (capped to the cell limit).
  - Resume-safe: rows whose eval_status is already "ok" are skipped; failed rows
    retry. Preflight verifies the DeepSeek deployment before any link is spent.

Column meaning note: c3_correctness is temporal_correctness on trajectory rows
and logical_correctness on recipe rows; c2_coverage is coverage vs
coverage_of_procedure. Same weights as the rubric slides.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("json_repair", "json-repair"),
                   ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _pip([_pkg])

import json, re, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import requests, json_repair, gspread
from openai import OpenAI

try:
    import trafilatura; _HAVE_TRAF = True
except Exception:
    _HAVE_TRAF = False

# =====================================================================
# 1. CONFIG
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_judge.txt"

SHEET_URL = "https://docs.google.com/spreadsheets/d/1g8WWS3Rli-4QAja6PO4JKPRGMAEVxr215A-vGPxFtlc/edit?usp=sharing"
TAB_NAME  = None
LINK_HEADER, GRAPH_HEADER, TYPE_HEADER = "link", "graph", "type"

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "DeepSeek-V4-Pro"
AZURE_API_KEY   = ""               # <-- paste your (regenerated) key

MAX_OUTPUT_TOK  = 8000
MAX_ARTICLE_CH  = 40000
HTTP_TIMEOUT    = 45
MAX_RETRIES     = 5
HNR_GATE        = 0.20
CELL_CAP        = 49000
SKIP_ALREADY_OK = True

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0"}

TRAJ_CRITERIA = ["grounding_faithfulness", "coverage", "temporal_correctness",
                 "structural_validity", "granularity_normalisation", "calibration_abstention"]
RECIPE_CRITERIA = ["grounding_faithfulness", "coverage_of_procedure", "logical_correctness",
                   "structural_validity", "granularity_normalisation", "calibration_abstention"]

EVAL_COLS = ["eval_status", "link_class", "on_type",
    "c1_grounding_faithfulness", "c2_coverage", "c3_correctness",
    "c4_structural_validity", "c5_granularity_normalisation", "c6_calibration_abstention",
    "composite_TQS_RQS", "gate_flags", "passed_gates", "over_reach", "graph_good",
    "criteria_notes_json", "coherence", "coherence_reason", "link_well_handled",
    "worst_offenders", "mech_json", "judgement_json"]

# =====================================================================
# 2. AUTH / SHEET
# =====================================================================
def open_ws():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    u = SHEET_URL.strip()
    if "PASTE_YOUR" in u:
        raise SystemExit("Set SHEET_URL in section 1.")
    sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    ws = sh.worksheet(TAB_NAME) if TAB_NAME else sh.get_worksheet(0)
    print(f"[sheet] opened '{sh.title}' / tab '{ws.title}'")
    return ws

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client():
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

def a1(idx0):
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

# =====================================================================
# 3. GRAPH PARSING + MECHANICAL (primitive-aware, cycle-legality fixed)
# =====================================================================
def parse_graph(cell):
    s = re.sub(r"^```(?:json)?|```$", "", str(cell or "").strip(), flags=re.M).strip()
    if not s.startswith("{"):
        return None
    try:
        g = json.loads(s)
    except Exception:
        try:
            g = json_repair.loads(s)
        except Exception:
            return None
    return g if isinstance(g, dict) else None

def _norm(s):
    s = unicodedata.normalize("NFKD", s or "")
    s = (s.replace("\u2019","'").replace("\u2018","'").replace("\u201c",'"')
           .replace("\u201d",'"').replace("\u2013","-").replace("\u2014","-"))
    return re.sub(r"\s+", " ", s).strip().lower()

def node_type(n):
    return str(n.get("primitive") or n.get("type") or "?")

def edge_ends(e):
    return e.get("source", e.get("from")), e.get("target", e.get("to"))

def _grounding(nodes, art):
    ung = [n.get("id") for n in nodes
           if not (lambda q: q and len(q) >= 10 and q in art)(_norm(str(n.get("evidence",""))))]
    egr = round(1 - len(ung)/len(nodes), 3) if nodes else 0.0
    return egr, round(1-egr, 3) if nodes else 1.0, ung

def _back_edges(nodes, edges):
    ids = {n.get("id") for n in nodes}
    adj = defaultdict(list)
    for e in edges:
        s, t = edge_ends(e)
        if s in ids and t in ids: adj[s].append(t)
    color, back = {i: 0 for i in ids}, []
    def dfs(u):
        color[u] = 1
        for v in adj[u]:
            if color.get(v) == 1: back.append((u, v))
            elif color.get(v) == 0: dfs(v)
        color[u] = 2
    for i in list(ids):
        if color[i] == 0: dfs(i)
    return back

FP_PAST = re.compile(r"\b(i|we)\s+(was|were|did|applied|emailed|built|moved|quit|started|"
                     r"finished|joined|got|went|paid|lost|wrote|submitted|received|passed)\b", re.I)
LEGAL_PRIMS = {"Op","Branch","AND","OR","NOT","Repeat"}

def mech_traj(g, art):
    nodes, edges = g.get("nodes", []) or [], g.get("edges", []) or []
    egr, hnr, ung = _grounding(nodes, art)
    types = Counter(node_type(n) for n in nodes)
    back = _back_edges(nodes, edges)
    m = {"n_nodes": len(nodes), "n_edges": len(edges), "EGR": egr, "HNR": hnr,
         "ungrounded_ids": ung[:10], "has_cycle": bool(back),
         "start_state_count": types.get("start_state", 0),
         "outcome_count": types.get("outcome", 0)}
    gates = []
    if hnr > HNR_GATE: gates.append(f"HNR>{HNR_GATE}")
    if m["has_cycle"]: gates.append("cycle")
    if m["start_state_count"] != 1: gates.append("start_state_count!=1")
    if m["outcome_count"] < 1: gates.append("missing_outcome")
    m["gates"] = gates
    return m

def mech_recipe(g, art):
    nodes, edges = g.get("nodes", []) or [], g.get("edges", []) or []
    egr, hnr, ung = _grounding(nodes, art)
    node_by = {n.get("id"): n for n in nodes}
    back = _back_edges(nodes, edges)
    repeat_out = {edge_ends(e) for e in edges
                  if node_type(node_by.get(edge_ends(e)[0], {})) == "Repeat"
                  and str(node_by.get(edge_ends(e)[0], {}).get("stop_condition","")).strip()}
    residual = _back_edges(nodes, [e for e in edges if edge_ends(e) not in repeat_out])
    legal = bool(back) and not residual
    illegal = sorted({node_type(n) for n in nodes} - LEGAL_PRIMS - {"?"})
    reps = [i for i, x in node_by.items() if node_type(x) == "Repeat"]
    rep_no_stop = [i for i in reps if not str(node_by[i].get("stop_condition","")).strip()]
    br_issues = []
    for i, x in node_by.items():
        if node_type(x) != "Branch": continue
        outs = [e for e in edges if edge_ends(e)[0] == i]
        conds = [_norm(str(e.get("condition",""))) for e in outs]
        if len(outs) < 2: br_issues.append(f"{i}:<2edges")
        if not any(c in ("else","otherwise","fallback","default") for c in conds):
            br_issues.append(f"{i}:no_fallback")
    leak = [n.get("id") for n in nodes
            if FP_PAST.search(f"{n.get('label','')} {n.get('description','')}")]
    m = {"n_nodes": len(nodes), "n_edges": len(edges), "EGR": egr, "HNR": hnr,
         "ungrounded_ids": ung[:10], "has_cycle": bool(back),
         "legal_repeat_cycle": legal, "illegal_primitives": illegal,
         "untyped_nodes": sum(1 for n in nodes if node_type(n) == "?"),
         "ungated_edges": sum(1 for e in edges if not str(e.get("condition","")).strip()),
         "repeats_missing_stop": rep_no_stop, "branch_issues": br_issues,
         "personal_log_leak_ids": leak[:10]}
    gates = []
    if illegal: gates.append("illegal_primitives")
    if hnr > HNR_GATE: gates.append(f"ungrounded>{HNR_GATE}")
    if m["has_cycle"] and not legal: gates.append("cycle_not_repeat_gated")
    if br_issues: gates.append("branch_missing_fallback_or_arity")
    if rep_no_stop: gates.append("repeat_missing_stop_condition")
    if len(leak) > 2: gates.append("personal_log_leakage")
    m["gates"] = gates
    return m

# =====================================================================
# 4. FETCH ARTICLE (direct -> Jina)
# =====================================================================
def fetch_article(url):
    text = ""
    try:
        r = requests.get(url, headers=UA, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        if _HAVE_TRAF:
            text = trafilatura.extract(r.text, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        if not text:
            text = re.sub(r"<[^>]+>", " ", r.text)
    except Exception:
        text = ""
    if len(text.strip()) < 400:
        try:
            r = requests.get("https://r.jina.ai/" + url,
                             headers={**UA, "X-Return-Format": "text"},
                             timeout=HTTP_TIMEOUT * 2)
            r.raise_for_status()
            if len(r.text.strip()) > len(text.strip()):
                text = r.text
        except Exception:
            pass
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", re.sub(r"[ \t]+", " ", text)).strip()
    return text[:MAX_ARTICLE_CH]

# =====================================================================
# 5. JUDGE CALL + PARSE
# =====================================================================
def call_judge(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        for tok_param in ("max_completion_tokens", "max_tokens"):
            try:
                resp = client.chat.completions.create(
                    model=DEPLOYMENT_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    **{tok_param: MAX_OUTPUT_TOK})
                return resp.choices[0].message.content or ""
            except Exception as exc:
                last = exc
                if "max_completion_tokens" in str(exc) or "max_tokens" in str(exc):
                    continue
                break
        if attempt < MAX_RETRIES:
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(str(last))

def parse_judgement(raw):
    b = re.sub(r"<think>.*?</think>", "", raw.strip(), flags=re.S)
    m = re.search(r"<judgement>(.*?)</judgement>", b, re.S | re.I)
    b = m.group(1).strip() if m else (re.search(r"\{.*\}", b, re.S).group(0)
                                      if re.search(r"\{.*\}", b, re.S) else b)
    b = re.sub(r"^```(?:json)?|```$", "", b, flags=re.M).strip()
    try:
        return json.loads(b)
    except Exception:
        try:
            j = json_repair.loads(b)
            return j if isinstance(j, dict) else None
        except Exception:
            return None

# =====================================================================
# 6. ROW OUTPUT
# =====================================================================
def row_values(gtype, label, j, mech_one, status="ok"):
    """Flatten this row's axis of the judgement into EVAL_COLS order."""
    out = {c: "" for c in EVAL_COLS}
    out["eval_status"] = status
    out["link_class"] = label
    out["mech_json"] = json.dumps(mech_one, ensure_ascii=False)[:CELL_CAP] if mech_one else ""
    if j is None:
        return [str(out[c]) for c in EVAL_COLS]
    axis = j.get("traj") if gtype == "trajectory" else j.get("recipe")
    criteria = TRAJ_CRITERIA if gtype == "trajectory" else RECIPE_CRITERIA
    if isinstance(axis, dict):
        crit = axis.get("criteria", {}) or {}
        keys = ["c1_grounding_faithfulness", "c2_coverage", "c3_correctness",
                "c4_structural_validity", "c5_granularity_normalisation",
                "c6_calibration_abstention"]
        notes = {}
        for col, cname in zip(keys, criteria):
            entry = crit.get(cname, {}) or {}
            out[col] = entry.get("score", "")
            notes[cname] = {"note": str(entry.get("note",""))[:120],
                            "span": str(entry.get("span",""))[:200],
                            "ids": (entry.get("ids") or [])[:5]}
        out["criteria_notes_json"] = json.dumps(notes, ensure_ascii=False)[:CELL_CAP]
        out["on_type"] = axis.get("on_type", "")
        out["composite_TQS_RQS"] = axis.get("TQS", axis.get("RQS", ""))
        out["gate_flags"] = ";".join(map(str, axis.get("gate_flags", [])))
        out["passed_gates"] = axis.get("passed_gates", "")
        out["over_reach"] = axis.get("over_reach", "")
        v = j.get("verdict", {}) or {}
        out["graph_good"] = v.get("traj_good" if gtype == "trajectory" else "recipe_good", "")
    out["coherence"] = j.get("coherence", "")
    out["coherence_reason"] = str(j.get("coherence_reason", ""))[:300]
    out["link_well_handled"] = (j.get("verdict", {}) or {}).get("link_well_handled", "")
    out["worst_offenders"] = " | ".join(map(str, j.get("worst_offenders", [])))[:500]
    out["judgement_json"] = json.dumps(j, ensure_ascii=False)[:CELL_CAP]
    return [str(out[c]) for c in EVAL_COLS]

# =====================================================================
# 7. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8")
    client = build_client()

    # preflight
    ok = False
    for tok in ({"max_tokens": 800}, {"max_completion_tokens": 800}):
        try:
            client.chat.completions.create(model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": "reply with: ok"}], **tok)
            ok = True; break
        except Exception as exc:
            last = exc
    if not ok:
        raise SystemExit(f"[preflight] '{DEPLOYMENT_NAME}' failed: {str(last)[:200]}")
    print(f"[preflight] {DEPLOYMENT_NAME} responds")

    ws = open_ws()
    rows = ws.get_all_values()
    header = [h.strip().lower() for h in rows[0]]
    li = header.index(LINK_HEADER) if LINK_HEADER in header else 0
    gi = header.index(GRAPH_HEADER)
    ty = header.index(TYPE_HEADER)

    # eval columns: reuse existing block if headers already written, else append
    if "eval_status" in header:
        start_col = header.index("eval_status")
        print(f"[cols] found existing eval columns at {a1(start_col)}")
    else:
        start_col = len(rows[0])
        ws.update(f"{a1(start_col)}1", [EVAL_COLS], value_input_option="RAW")
        print(f"[cols] eval columns start at {a1(start_col)}")

    # group rows by link
    links = defaultdict(dict)   # link -> {"trajectory": (row_idx, graph), "recipe": (...)}
    for idx, r in enumerate(rows[1:], start=2):   # sheet row numbers (1-based, +header)
        link = (r[li] if li < len(r) else "").strip()
        if not link: continue
        gtype = (r[ty] if ty < len(r) else "").strip().lower()
        gtype = "trajectory" if gtype.startswith("traj") else ("recipe" if gtype.startswith("rec") else gtype)
        if gtype not in ("trajectory", "recipe"): continue
        g = parse_graph(r[gi] if gi < len(r) else "")
        if g is not None and "error" in g and not g.get("nodes"):
            g = None
        st = (r[start_col] if start_col < len(r) else "").strip()
        links[link][gtype] = {"row": idx, "graph": g if (g and g.get("nodes")) else None,
                              "done": st == "ok"}

    todo = []
    for link, parts in links.items():
        if SKIP_ALREADY_OK and all(p["done"] for p in parts.values()):
            continue
        if any(p["graph"] for p in parts.values()):
            todo.append(link)
    print(f"[plan] {len(links)} links | to evaluate: {len(todo)} | "
          f"mixed: {sum(1 for p in links.values() if len(p) == 2)}")

    def write_rows(pairs):
        """pairs: list of (sheet_row, values). Retry on quota."""
        delay = 2.0
        for attempt in range(6):
            try:
                ws.batch_update([{"range": f"{a1(start_col)}{r}:{a1(start_col+len(EVAL_COLS)-1)}{r}",
                                  "values": [vals]} for r, vals in pairs],
                                value_input_option="RAW")
                return
            except Exception as exc:
                if "429" in str(exc) or "quota" in str(exc).lower():
                    time.sleep(delay); delay = min(delay * 2, 30); continue
                print(f"  !! write failed: {str(exc)[:100]}"); return

    counts = {}
    for n, link in enumerate(sorted(todo), 1):
        parts = links[link]
        g_t = parts.get("trajectory", {}).get("graph")
        g_r = parts.get("recipe", {}).get("graph")
        label = "mixed" if (("trajectory" in parts) and ("recipe" in parts)) else \
                ("trajectory" if "trajectory" in parts else "recipe")
        art = fetch_article(link)
        if len(art) < 400:
            write_rows([(p["row"], row_values(gt, label, None, None, "article_fetch_failed"))
                        for gt, p in parts.items()])
            counts["fetch_failed"] = counts.get("fetch_failed", 0) + 1
            print(f"[{n}/{len(todo)}] fetch_failed  {link[:70]}"); continue
        art_norm = _norm(art)
        mech = {"traj": mech_traj(g_t, art_norm) if g_t else None,
                "recipe": mech_recipe(g_r, art_norm) if g_r else None}
        prompt = (template
                  .replace("{{ARTICLE_TEXT}}", art)
                  .replace("{{LINK_LABEL}}", label)
                  .replace("{{MECHANICAL_JSON}}", json.dumps(mech, ensure_ascii=False))
                  .replace("{{TRAJECTORY_GRAPH_JSON}}", json.dumps(g_t, ensure_ascii=False) if g_t else "null")
                  .replace("{{RECIPE_GRAPH_JSON}}", json.dumps(g_r, ensure_ascii=False) if g_r else "null")
                  .replace("{{URL}}", link))
        try:
            raw = call_judge(client, prompt)
        except Exception as exc:
            write_rows([(p["row"], row_values(gt, label, None,
                        mech["traj" if gt == "trajectory" else "recipe"],
                        f"api_failed: {str(exc)[:120]}")) for gt, p in parts.items()])
            counts["api_failed"] = counts.get("api_failed", 0) + 1
            print(f"[{n}/{len(todo)}] api_failed  {str(exc)[:80]}")
            if counts["api_failed"] >= 3 and counts.get("ok", 0) == 0:
                print("[abort] first 3 calls failed - fix config first"); break
            continue
        j = parse_judgement(raw)
        if j is None:
            write_rows([(p["row"], row_values(gt, label, None,
                        mech["traj" if gt == "trajectory" else "recipe"],
                        "parse_failed")) for gt, p in parts.items()])
            counts["parse_failed"] = counts.get("parse_failed", 0) + 1
            print(f"[{n}/{len(todo)}] parse_failed  {link[:70]}"); continue
        write_rows([(p["row"], row_values(gt, label, j,
                    mech["traj" if gt == "trajectory" else "recipe"]))
                    for gt, p in parts.items()])
        counts["ok"] = counts.get("ok", 0) + 1
        t, r_ = j.get("traj") or {}, j.get("recipe") or {}
        print(f"[{n}/{len(todo)}] ok  TQS={t.get('TQS','-')} RQS={r_.get('RQS','-')} "
              f"coh={j.get('coherence','-')}  {link[:55]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] labels written next to each row, columns {a1(start_col)} onward")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[preflight] DeepSeek-V4-Pro responds
[sheet] opened 'evaluation' / tab 'Sheet1'
[cols] found existing eval columns at D
[plan] 118 links | to evaluate: 1 | mixed: 118
[1/1] fetch_failed  https://cs.utdallas.edu/30355/interview-with-cs-phd-alumni-series-mary

[done] fetch_failed=1
[done] labels written next to each row, columns D onward
